# End-to-End Pipeline: synth_kit → geospatial → entity_resolution → feature_engineering

This notebook demonstrates a realistic healthcare analytics workflow using all four complete modules:

| Step | Module | What happens |
|------|--------|--------------|
| 1 | `synth_kit` | PHI in a member demographics table is replaced — names, DOBs, and addresses are synthesized while schema and distributions are preserved |
| 2 | `geospatial` | Synthetic addresses are geocoded to Census block groups — a useful SDoH proxy |
| 3 | `entity_resolution` | Member records from three simulated source systems are linked to a common patient identity using weighted fuzzy matching and NetworkX graph clustering |
| 4 | `feature_engineering` | Windowed pharmacy and medical utilization features are computed for the synthetic population |
| 5 | — | A final join produces the ML-ready dataset: demographics + geography + features |

**Geocoding prerequisite** (one-time setup): run `analytics-toolbox ingest-tiger` and `analytics-toolbox ingest-nad` to download public reference data. The notebook detects whether this has been done and falls back gracefully if not.

## 1. Setup

In [1]:
import os
import random
import tempfile
import warnings
from pathlib import Path

import duckdb
import pandas as pd
from sqlalchemy import create_engine

from analytics_toolbox.feature_engineering import join_features
from analytics_toolbox.feature_engineering.examples.medicaid_features import (
    med_utilization_features,
    rx_spend_features,
)
from analytics_toolbox.synth_kit import synthesize

SEED = 42
N_MEMBERS = 60
AS_OF = pd.Timestamp("2024-06-30")

# Resolve config.yaml from CWD or one level up — handles both repo-root launch
# and VS Code / JupyterLab where the kernel CWD is set to the notebook's directory.
_candidates = [Path.cwd() / "config.yaml", Path.cwd().parent / "config.yaml"]
CONFIG_PATH = next((str(p) for p in _candidates if p.exists()), "config.yaml")

print("Imports OK.")
if Path(CONFIG_PATH).exists():
    print(f"Config: {CONFIG_PATH}")
else:
    print("Config: not found (geocoding will be skipped)")

Imports OK.
Config: /Users/matthewbargstadt/code/analytics-toolbox/notebooks/config.yaml


## 2. Source Data — Simulating a Real System

We construct a member demographics table that represents what an operational system might contain: real names, dates of birth, and addresses. In practice this would be a production database.

This data must **never** leave the secure environment — which is exactly what `synth_kit` solves in the next section.

In [2]:
rng = random.Random(SEED)

FIRST_NAMES = ["James", "Mary", "Robert", "Patricia", "John",
               "Jennifer", "Michael", "Linda", "David", "Barbara"]
LAST_NAMES  = ["Smith", "Johnson", "Williams", "Brown", "Jones",
               "Garcia", "Miller", "Davis", "Wilson", "Taylor"]
STREETS     = ["Main St", "Oak Ave", "Elm Dr", "Park Rd",
               "Cedar Blvd", "First St", "Maple Ln", "Washington Ave"]
CITIES      = [
    ("Des Moines",   "IA", "50309"),
    ("Ames",         "IA", "50010"),
    ("Cedar Rapids", "IA", "52401"),
    ("Iowa City",    "IA", "52240"),
    ("Davenport",    "IA", "52801"),
]
ELIGIBILITY = ["adult", "aged", "child", "blind_disabled", "pregnant"]

source_rows = []
for mid in range(1001, 1001 + N_MEMBERS):
    city, state, zip_code = rng.choice(CITIES)
    dob = f"19{rng.randint(40, 89):02d}-{rng.randint(1, 12):02d}-{rng.randint(1, 28):02d}"
    source_rows.append({
        "member_id":            mid,
        "first_name":           rng.choice(FIRST_NAMES),
        "last_name":            rng.choice(LAST_NAMES),
        "dob":                  dob,
        "street_address":       f"{rng.randint(100, 9999)} {rng.choice(STREETS)}",
        "city":                 city,
        "state":                state,
        "zip_code":             zip_code,
        "eligibility_category": rng.choice(ELIGIBILITY),
        "enrollment_months":    rng.randint(3, 36),
    })

members_source = pd.DataFrame(source_rows)

_tmp_db = tempfile.mktemp(suffix=".duckdb")
_tmp_con = duckdb.connect(_tmp_db)
_tmp_con.register("_m", members_source)
_tmp_con.execute("CREATE TABLE member_demographics AS SELECT * FROM _m")
_tmp_con.close()

print(f"Source table: {len(members_source)} members with PHI")
members_source[["member_id", "first_name", "last_name", "dob",
                "street_address", "city", "state", "zip_code"]].head()

Source table: 60 members with PHI


,member_id,first_name,last_name,dob,street_address,city,state,zip_code
0,1001,Patricia,Brown,1941-12-09,2386 Oak Ave,Des Moines,IA,50309
1,1002,Mary,Brown,1967-01-01,3911 Main St,Davenport,IA,52801
2,1003,Barbara,Jones,1966-04-15,206 Elm Dr,Davenport,IA,52801
3,1004,Jennifer,Johnson,1949-04-25,1619 Maple Ln,Cedar Rapids,IA,52401
4,1005,James,Davis,1978-05-26,8885 Oak Ave,Cedar Rapids,IA,52401


## 3. synth_kit — PHI Replacement

`synthesize()` profiles the table server-side (SQL aggregates only — raw rows never enter Python memory), then generates a fully synthetic version:

- **PHI columns** (`first_name`, `last_name`, `dob`, `street_address`) are replaced with Faker-generated values.
- **Categorical columns** (`city`, `state`, `zip_code`, `eligibility_category`) are sampled from the source distribution.
- **Numeric columns** (`enrollment_months`) are drawn from a fitted statistical distribution.
- `member_id` is excluded from synthesis so we can join downstream; fresh sequential IDs are assigned instead.

The result is safe to share outside the secure environment — no real PHI survives.

In [3]:
engine = create_engine(f"duckdb:///{_tmp_db}")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    synth_raw = synthesize(
        engine,
        """
        SELECT first_name, last_name, dob,
               street_address, city, state, zip_code,
               eligibility_category, enrollment_months
        FROM member_demographics
        """,
        n_rows=N_MEMBERS,
        random_seed=SEED,
        # When config.yaml is present and NAD is ingested, synthesize() draws
        # street addresses from real NAD data for the dominant state rather
        # than Faker-generated strings.
        geospatial_config=CONFIG_PATH if Path(CONFIG_PATH).exists() else None,
    )

synth_raw.insert(0, "member_id", range(2001, 2001 + len(synth_raw)))
synthetic_members = synth_raw.copy()
os.unlink(_tmp_db)

print(f"Synthetic members: {synthetic_members.shape}")
print("PHI replaced — original vs synthetic:")
print(f"  Source first_name[0]:    {members_source.iloc[0]['first_name']}")
print(f"  Synthetic first_name[0]: {synthetic_members.iloc[0]['first_name']}")
synthetic_members[["member_id", "first_name", "last_name", "dob",
                   "street_address", "city", "state", "zip_code"]].head()

Synthetic members: (60, 10)
PHI replaced — original vs synthetic:
  Source first_name[0]:    Patricia
  Synthetic first_name[0]: Danielle


,member_id,first_name,last_name,dob,street_address,city,state,zip_code
0,2001,Danielle,Mayo,2002-05-27,1111 E 14TH ST,Davenport,IA,52401
1,2002,Angel,Bowman,1979-11-16,155 SW MAPLEWOOD DR,Ames,IA,50010
2,2003,Joshua,Hawkins,2004-02-16,1320 E 25TH ST,Des Moines,IA,52401
3,2004,Jeffrey,Cohen,1979-01-08,1531 INDIANA AVE,Des Moines,IA,52240
4,2005,Jill,Cameron,1925-07-02,1010 FREMONT ST,Des Moines,IA,52801


## 4. Geospatial — Address Geocoding

The geospatial pipeline normalizes synthetic addresses per USPS Pub 28, fuzzy-matches them against the National Address Database (NAD), and assigns Census block group FIPS codes via point-in-polygon against TIGER shapefiles.

Block group FIPS is a strong SDoH proxy: it links to ACS demographic, income, and housing attributes at a sub-ZIP level.

**Prerequisites** (run once to download public reference data):
```bash
analytics-toolbox ingest-tiger --config config.yaml
analytics-toolbox ingest-nad   --config config.yaml
```

If these haven’t been run, the cell falls back gracefully and geocoding columns are `None`.

In [4]:
GEOCODED = False
geocoded_members = synthetic_members.copy()

address_input = synthetic_members[
    ["member_id", "street_address", "city", "state", "zip_code"]
].rename(
    columns={
        "street_address": "Street_Address",
        "city":           "City",
        "state":          "State",
        "zip_code":       "Postal_Code",
    }
)

if Path(CONFIG_PATH).exists():
    try:
        from analytics_toolbox.geospatial import geocode_address_table
        geo_result = geocode_address_table(address_input, CONFIG_PATH)
        geocoded_members = synthetic_members.merge(
            geo_result[["member_id", "block_group_fips",
                        "matched_latitude", "matched_longitude",
                        "location_imputed", "match_score"]],
            on="member_id", how="left",
        )
        GEOCODED = True
        n_assigned = geocoded_members["block_group_fips"].notna().sum()
        print(f"Geocoded: {n_assigned} / {len(geocoded_members)} members assigned a block group")
    except Exception as e:
        print(f"Geocoding unavailable: {e}")
        print("Continuing without block group assignment.")
else:
    print("No config.yaml found — geospatial step skipped.")
    print("To enable: run 'analytics-toolbox ingest-tiger' and 'analytics-toolbox ingest-nad'.")

if not GEOCODED:
    for col in ["block_group_fips", "matched_latitude", "matched_longitude", "location_imputed"]:
        geocoded_members[col] = None

geocoded_members[["member_id", "city", "state", "zip_code",
                  "block_group_fips", "matched_latitude", "matched_longitude"]].head()

Geocoded: 60 / 60 members assigned a block group


,member_id,city,state,zip_code,block_group_fips,matched_latitude,matched_longitude
0,2001,Davenport,IA,52401,191130027002,41.975484,-91.658769
1,2002,Ames,IA,50010,191690009003,42.033223,-93.587622
2,2003,Des Moines,IA,52401,191130027002,41.975484,-91.658769
3,2004,Des Moines,IA,52240,191030018012,41.631536,-91.499230
4,2005,Des Moines,IA,52801,191630109003,41.520770,-90.574727


## 5. Entity Resolution — Multi-System Patient Linking

In healthcare analytics, patient records are scattered across independent source systems — Medicaid enrollment, hospital EHRs, pharmacy PBMs — each assigning their own internal IDs. Linking these to a common patient identity is essential before any population-level analysis.

`resolve()` links records across systems in three stages:

1. **Block** — candidates are grouped by date of birth (primary) or last name + postal code (secondary, for null DOBs), keeping comparisons O(block size) not O(N²)
2. **Score** — weighted RapidFuzz similarity across name, address, and demographic fields produces a single confidence score per candidate pair
3. **Cluster** — matched pairs form an undirected graph; NetworkX connected components identify the final entity clusters

**HIPAA compliance**: no PHI appears in log messages or output — the cluster DataFrame contains only system IDs and similarity scores.

In [ ]:
import dataclasses

from analytics_toolbox._config import AnalyticsToolboxConfig, load_config
from analytics_toolbox.entity_resolution import resolve
from analytics_toolbox.entity_resolution._config import EntityResolutionConfig


def _make_er_system(source_df: pd.DataFrame, id_col: str, id_prefix: str) -> pd.DataFrame:
    """Rename synthetic member columns to ER canonical names and assign system IDs."""
    source_df = source_df.reset_index(drop=True)
    out = source_df.rename(columns={
        "dob":            "DOB",
        "last_name":      "Last_Name",
        "first_name":     "First_Name",
        "street_address": "Address",
        "city":           "City",
        "state":          "State",
        "zip_code":       "Postal_Code",
    }).copy()
    out[id_col] = [f"{id_prefix}{mid}" for mid in source_df["member_id"]]
    keep = [id_col, "DOB", "Last_Name", "First_Name", "Address", "City", "State", "Postal_Code"]
    return out[[c for c in keep if c in out.columns]]


# Three simulated source systems — all drawn from the same 60-member synthetic population.
# In a real deployment each would be a separate database pull (Medicaid, hospital, pharmacy PBM).
#
# Note: synth_kit already emitted NAD-normalized street addresses, and resolve() re-runs
# scourgify normalization internally. That re-normalization is defensive and intentional —
# entity_resolution never assumes its inputs were pre-normalized, so it works on raw input
# pulls too. The duplicate pass here is cheap and harmless.
system_a = _make_er_system(synthetic_members, "medicaid_id", "MCD-")
system_b = _make_er_system(synthetic_members.sample(40, random_state=SEED), "hospital_id", "HOSP-")
system_c = _make_er_system(synthetic_members.sample(30, random_state=SEED + 1), "pbm_id", "PBM-")

systems = {"medicaid_id": system_a, "hospital_id": system_b, "pbm_id": system_c}

# Build config — inherit storage + geospatial from config.yaml if present; inject ER defaults.
if Path(CONFIG_PATH).exists():
    _base = load_config(CONFIG_PATH)
else:
    from analytics_toolbox.geospatial._config import StorageConfig
    _base = AnalyticsToolboxConfig(
        storage=StorageConfig(data_dir=Path("/tmp"), connection="/tmp/er_demo.duckdb"),
        geospatial=None,
    )

er_config = dataclasses.replace(
    _base,
    entity_resolution=_base.entity_resolution or EntityResolutionConfig(match_threshold=0.70),
)

clusters = resolve(systems, config=er_config)

system_cols = ["medicaid_id", "hospital_id", "pbm_id"]
n_multi = (clusters[system_cols].notna().sum(axis=1) > 1).sum()
avg_cluster_size = clusters[system_cols].notna().sum(axis=1).mean()

print(f"Total clusters:        {len(clusters)}")
print(f"Multi-system clusters: {n_multi}  (record linked across ≥2 source systems)")
print(f"Avg records/cluster:   {avg_cluster_size:.2f}")
print(f"Avg similarity (mean): {clusters['avg_similarity'].mean():.3f}")
clusters.head(10)

## 6. Claims Data for the Synthetic Population

We generate synthetic pharmacy and medical claims for the new member IDs (2001–2060). In a real pipeline these would be synthesized alongside the member table — or joined from production claims after the anonymization step.

For this demo, we generate them directly to keep the notebook self-contained.

In [6]:
synth_rng = random.Random(SEED + 1)

DRUG_CLASSES = ["analgesic", "antibiotic", "antidiabetic", "cardiovascular", "psychiatric"]
PLACES       = ["inpatient", "outpatient", "ed", "office", "home"]
PROVIDERS    = ["physician", "nurse_practitioner", "specialist", "hospital"]
START        = pd.Timestamp("2023-01-01")
SPAN_DAYS    = (AS_OF - START).days

rx_rows, med_rows = [], []
for mid in synthetic_members["member_id"]:
    for _ in range(synth_rng.randint(0, 18)):
        dt = START + pd.Timedelta(days=synth_rng.randint(0, SPAN_DAYS))
        rx_rows.append({
            "claim_id":   len(rx_rows) + 1,
            "member_id":  mid,
            "claim_date": dt.date(),
            "ndc_code":   f"NDC{synth_rng.randint(10000, 99999)}",
            "drug_class": synth_rng.choice(DRUG_CLASSES),
            "paid_amount":round(synth_rng.uniform(5.0, 500.0), 2),
            "days_supply":synth_rng.randint(1, 90),
            "quantity":   synth_rng.randint(1, 180),
        })
    for _ in range(synth_rng.randint(0, 8)):
        dt = START + pd.Timedelta(days=synth_rng.randint(0, SPAN_DAYS))
        med_rows.append({
            "claim_id":         len(med_rows) + 1,
            "member_id":        mid,
            "claim_date":       dt.date(),
            "place_of_service": synth_rng.choice(PLACES),
            "provider_type":    synth_rng.choice(PROVIDERS),
            "dx_primary":       f"Z{synth_rng.randint(100, 999)}",
            "paid_amount":      round(synth_rng.uniform(50.0, 5000.0), 2),
        })

rx_claims  = pd.DataFrame(rx_rows)
med_claims = pd.DataFrame(med_rows)
print(f"rx_claims:  {len(rx_claims):,} rows")
print(f"med_claims: {len(med_claims):,} rows")
rx_claims.head(3)

rx_claims:  569 rows
med_claims: 272 rows


,claim_id,member_id,claim_date,ndc_code,drug_class,paid_amount,days_supply,quantity
0,1,2001,2023-10-20,NDC28865,cardiovascular,188.09,90,25
1,2,2002,2023-04-07,NDC91624,cardiovascular,394.51,43,85
2,3,2003,2023-05-02,NDC79775,cardiovascular,133.20,90,117


## 7. Feature Engineering — Windowed Utilization Features

We load the synthetic claims into a DuckDB connection and compute windowed features on an entity-date spine. The engine uses a single SQL scan with `FILTER` clauses to produce all window aggregates in one pass — no per-window re-scans.

In [7]:
con = duckdb.connect()
con.register("_rx",  rx_claims)
con.register("_med", med_claims)
con.execute("CREATE TABLE rx_claims  AS SELECT * FROM _rx")
con.execute("CREATE TABLE med_claims AS SELECT * FROM _med")

as_of_dates = [pd.Timestamp("2024-03-31"), AS_OF]
member_ids  = synthetic_members["member_id"].tolist()

spine = pd.DataFrame(
    [{"member_id": mid, "as_of_date": dt} for dt in as_of_dates for mid in member_ids]
)
print(f"Spine: {spine.shape}  ({len(member_ids)} members × {len(as_of_dates)} snapshots)")

rx_features  = rx_spend_features(spine, con, windows=(30, 90, 365))
med_features = med_utilization_features(spine, con, windows=(30, 90, 365))
all_features = join_features([rx_features, med_features], on=["member_id", "as_of_date"])

feat_cols = [c for c in all_features.columns if c not in ["member_id", "as_of_date"]]
print(f"Features: {all_features.shape}  ({len(feat_cols)} feature columns)")
all_features.head(3)

Spine: (120, 2)  (60 members × 2 snapshots)
Features: (120, 29)  (27 feature columns)


,member_id,as_of_date,rx__paid_sum_30d,rx__claims_cnt_30d,rx__ndc_ndist_30d,rx__dsupply_sum_30d,rx__paid_sum_90d,rx__claims_cnt_90d,rx__ndc_ndist_90d,rx__dsupply_sum_90d,...,med__claims_cnt_90d,med__paid_sum_90d,med__ed_visits_cnt_90d,med__inpatient_cnt_90d,med__provider_ndist_90d,med__claims_cnt_365d,med__paid_sum_365d,med__ed_visits_cnt_365d,med__inpatient_cnt_365d,med__provider_ndist_365d
0,2001,2024-03-31,NaN,0,0,NaN,NaN,0,0,NaN,...,1,922.13,0.0,0.0,1,3,5523.27,0.0,2.0,2
1,2002,2024-03-31,NaN,0,0,NaN,NaN,0,0,NaN,...,3,5343.67,0.0,0.0,2,6,16988.44,0.0,1.0,3
2,2003,2024-03-31,NaN,0,0,NaN,381.47,2,2,101.0,...,1,1342.64,0.0,1.0,1,6,7944.98,0.0,3.0,4


## 8. Compose Final Dataset

Join the computed features with geocoded member demographics. The result is ML-ready: privacy-safe, spatially enriched, and feature-rich.

In [8]:
demog_keep = ["member_id", "eligibility_category", "enrollment_months",
              "city", "state", "zip_code", "block_group_fips"]
if GEOCODED:
    demog_keep += ["matched_latitude", "matched_longitude"]

member_attrs = geocoded_members[[c for c in demog_keep if c in geocoded_members.columns]]
final = all_features.merge(member_attrs, on="member_id", how="left")

rx_active = all_features["rx__claims_cnt_365d"].notna().sum()
med_active = all_features["med__claims_cnt_365d"].notna().sum()

print(f"Final dataset: {final.shape}")
print(f"  {len(as_of_dates)} snapshots × {N_MEMBERS} members = {len(as_of_dates) * N_MEMBERS} rows")
print(f"  {len(feat_cols)} feature columns + {len(demog_keep) - 1} demographic columns")
print(f"\nRX activity (365d):  {rx_active} / {len(all_features)} rows")
print(f"Med activity (365d): {med_active} / {len(all_features)} rows")

final

Final dataset: (120, 37)
  2 snapshots × 60 members = 120 rows
  27 feature columns + 8 demographic columns

RX activity (365d):  113 / 120 rows
Med activity (365d): 102 / 120 rows


,member_id,as_of_date,rx__paid_sum_30d,rx__claims_cnt_30d,rx__ndc_ndist_30d,rx__dsupply_sum_30d,rx__paid_sum_90d,rx__claims_cnt_90d,rx__ndc_ndist_90d,rx__dsupply_sum_90d,...,med__inpatient_cnt_365d,med__provider_ndist_365d,eligibility_category,enrollment_months,city,state,zip_code,block_group_fips,matched_latitude,matched_longitude
0,2001,2024-03-31,NaN,0,0,NaN,NaN,0,0,NaN,...,2.0,2,child,3.926226,Davenport,IA,52401,191130027002,41.975484,-91.658769
1,2002,2024-03-31,NaN,0,0,NaN,NaN,0,0,NaN,...,1.0,3,adult,31.566812,Ames,IA,50010,191690009003,42.033223,-93.587622
2,2003,2024-03-31,NaN,0,0,NaN,381.47,2,2,101.0,...,3.0,4,aged,31.538277,Des Moines,IA,52401,191130027002,41.975484,-91.658769
3,2004,2024-03-31,39.89,1,1,79.0,39.89,1,1,79.0,...,0.0,1,aged,33.121880,Des Moines,IA,52240,191030018012,41.631536,-91.499230
4,2005,2024-03-31,NaN,0,0,NaN,776.81,3,3,145.0,...,2.0,3,adult,23.850533,Des Moines,IA,52801,191630109003,41.520770,-90.574727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2048,2024-03-31,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,NaN,...,0.0,3,aged,8.487131,Davenport,IA,52401,191130027002,41.975484,-91.658769
116,2059,2024-03-31,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,NaN,...,3.0,3,adult,7.949047,Ames,IA,50309,191530051014,41.584478,-93.621560
117,2034,2024-06-30,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,NaN,...,0.0,1,pregnant,16.096366,Des Moines,IA,50010,191690009003,42.033223,-93.587622
118,2037,2024-06-30,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>,NaN,...,0.0,2,adult,8.765880,Cedar Rapids,IA,52801,191630109003,41.520770,-90.574727


In [9]:
print("=== 30-day RX spend (non-null rows) ===")
print(final["rx__paid_sum_30d"].describe().round(2))

if GEOCODED:
    print(f"\nBlock groups covered: {final['block_group_fips'].nunique()} unique")
else:
    print("\nBlock group: not geocoded (run ingest-tiger + ingest-nad to enable)")

=== 30-day RX spend (non-null rows) ===
count     40.00
mean     315.14
std      208.14
min       11.57
25%      131.72
50%      332.97
75%      455.85
max      812.06
Name: rx__paid_sum_30d, dtype: float64

Block groups covered: 5 unique


## Summary

| Step | Module | Key call | Output |
|------|--------|----------|--------|
| PHI replacement | `synth_kit` | `synthesize()` | Fake names/DOB/addresses, real distributions |
| Address geocoding | `geospatial` | `geocode_address_table()` | `block_group_fips`, `matched_latitude/longitude` per member |
| Patient linking | `entity_resolution` | `resolve()` | Cluster DataFrame: one row per entity, one column per source system ID |
| Windowed features | `feature_engineering` | `compute_features()` | 30/90/365-day RX spend + medical utilization |
| Composition | `feature_engineering` | `join_features()` | Single ML-ready frame |

**Next steps:**
- Load `final` into a modeling framework (scikit-learn, XGBoost) directly — grain is `(member_id, as_of_date)`, all feature columns are numeric.
- Enrich with ACS block-group attributes (income, education, housing) via the `block_group_fips` key.
- Extend `as_of_dates` for a time-series training set — earlier snapshots become training rows once future outcomes are known.
- Use `clusters` to join the master patient index across source systems before feature aggregation.